## **Download dependencies and libraries**

#Запускаем в случае ошибок импорта

In [ ]:
!pip install facebookads
!pip install pandas_datareader
!pip install facebook_business
!pip install tg_logger

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for facebookads: filename=facebookads-2.11.4-py3-none-any.whl size=526251 sha256=84e93a2dbd6b8e29d25a952aa72c54454202bce1b6db64bc87155ecac40d6002
  Stored in directory: /root/.cache/pip/wheels/3d/b5/4f/4c5366dfcc74cb4698ce4aadfd1c093449ebb7bba0f3fc50e0
Successfully built facebookads
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 64.4 MB/s eta 0:00:00
  Created wheel for curlify: filename=curlify-2.2.1-py3-none-any.whl size=2667 sha256=03c0d7bb67ef00d06e3820722e4d69adfa5b9c225d697342a54a262681e4161b
  Stored in directory: /root/.cache/pip/wheels/e1/6b/61/f8560ac125bd64f2b87b9af2f9ae08f8dbaec154f583a9e301
Successfully built curlify
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

#Инициализация

In [ ]:
import base64
import datetime
import logging
import time

import requests
from google.cloud import bigquery
from google.cloud.exceptions import NotFound
from facebook_business.adobjects.adaccount import AdAccount
from facebook_business.adobjects.adreportrun import AdReportRun
from facebook_business.adobjects.adsinsights import AdsInsights
from facebook_business.api import FacebookAdsApi
import tg_logger
from google.oauth2 import service_account

# **Define helper functions**

In [ ]:
logger = logging.getLogger()
logger.setLevel(logging.INFO)

schema_exchange_rate = [
    bigquery.SchemaField("date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("USDEUR", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("USDUAH", "FLOAT", mode="REQUIRED")
]
schema_facebook_stat = [
    bigquery.SchemaField("date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("account_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("ad_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("ad_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("adset_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("adset_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("campaign_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("campaign_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("clicks", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("impressions", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("spend", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField('conversions', 'RECORD', mode='REPEATED',
                         fields=(bigquery.SchemaField('action_type', 'STRING'),
                                 bigquery.SchemaField('value', 'STRING'))),
    bigquery.SchemaField('actions', 'RECORD', mode='REPEATED',
                         fields=(bigquery.SchemaField('action_type', 'STRING'),
                                 bigquery.SchemaField('value', 'STRING'))),
    bigquery.SchemaField('country', 'STRING', mode='NULLABLE')
]

schema_rows = [
    bigquery.SchemaField("date", "DATE"),
    bigquery.SchemaField("source", "STRING"),
    bigquery.SchemaField("customer_id", "STRING"),
    bigquery.SchemaField("table_id", "STRING"),
    bigquery.SchemaField("report", "STRING"),
    bigquery.SchemaField("num_rows", "INT64")
]

clustering_fields_facebook = ['campaign_id', 'campaign_name']


def get_date():
    '''
    Generates yesterday's date

    '''
    yesterday = datetime.datetime.now() - datetime.timedelta(1)

    logger.info('Date for reports was generated')

    return yesterday.strftime("%Y-%m-%d")


def exist_dataset_table(client, table_id, dataset_id, project_id, schema, clustering_fields=None):
    '''
    Validates and creates a table in Google BigQuery
    :client: Google BigQuery Client
    :table_id: table identifier
    :dataset_id: dataset identifier
    :project_id: project ID
    :schema: Google BigQuery table schema
    :clustering_fields: table clustering
    '''

    try:
        dataset_ref = f"{project_id}.{dataset_id}"
        client.get_dataset(dataset_ref)

    except NotFound:
        dataset_ref = f"{project_id}.{dataset_id}"
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "europe-central2"
        dataset = client.create_dataset(dataset)

        logger.info('Dataset was created %s.%s',
                    client.project, dataset.dataset_id)

    try:
        table_ref = f"{project_id}.{dataset_id}.{table_id}"

        client.get_table(table_ref)

    except NotFound:

        table_ref = f"{project_id}.{dataset_id}.{table_id}"

        table = bigquery.Table(table_ref, schema=schema)

        table.time_partitioning = bigquery.TimePartitioning(
            type_=bigquery.TimePartitioningType.DAY,
            field="date"
        )

        if clustering_fields is not None:
            table.clustering_fields = clustering_fields

        table = client.create_table(table)
        logger.info('Table was created %s', table_ref)

    return 'ok'


def bq_job_config(schema):
    '''
    Creates a config for uploading to Google BigQuery
    :schema: Google BigQuery table schema
    '''

    bq_config = bigquery.LoadJobConfig()
    bq_config.write_disposition = 'WRITE_APPEND'
    bq_config.source_format = "NEWLINE_DELIMITED_JSON"
    bq_config.schema = schema
    bq_config.autodetect = True
    logger.info('Job config was prepared')

    return bq_config


def insert_json_bq(client, table_id, dataset_id, project_id, data, schema):
    '''
    Uploads data to Google BigQuery table
    :client: Google BigQuery Client
    :table_id: table identifier
    :dataset_id: dataset identifier
    :project_id: project ID
    :data: ND JSON that has to be loaded
    :schema: Google BigQuery table schema
    '''

    job_config = bq_job_config(schema)

    table_ref = f"{project_id}.{dataset_id}.{table_id}"

    table = client.get_table(table_ref)

    resp = client.load_table_from_json(
        json_rows=data,
        destination=table_ref,
        job_config=job_config
    )

    resp.result()

    logger.info('Data was uploaded to the table %s', table.table_id)



# if pubsub_message == 'get_facebook':
#
#     table_id = event['attributes']['table_id']
#     dataset_id = event['attributes']['dataset_id']
#     project_id = event['attributes']['project_id']
#
#     app_id = event['attributes']['app_id']
#     app_secret = event['attributes']['app_secret']
#     access_token = event['attributes']['access_token']
#     account_id = event['attributes']['account_id']
#     country = event['attributes']['country']
#
#     try:
#         FacebookAdsApi.init(app_id, app_secret,
#                             access_token, api_version='v20.0')
#
#         account = AdAccount('act_' + str(account_id))
#
#         job = account.get_insights(fields=[
#             AdsInsights.Field.account_id,
#             AdsInsights.Field.campaign_id,
#             AdsInsights.Field.campaign_name,
#             AdsInsights.Field.adset_name,
#             AdsInsights.Field.adset_id,
#             AdsInsights.Field.ad_name,
#             AdsInsights.Field.ad_id,
#             AdsInsights.Field.spend,
#             AdsInsights.Field.impressions,
#             AdsInsights.Field.clicks,
#             AdsInsights.Field.actions,
#             AdsInsights.Field.conversions
#         ], params={
#             'level': 'ad',
#             'time_range': {
#                 'since': date_from,
#                 'until': date_to
#             },
#             'time_increment': 1
#         }, is_async=True)
#
#         while True:
#             job = job.api_get()
#             time.sleep(1)
#             if job[AdReportRun.Field.async_status] == 'Job Completed':
#                 logger.info("Async job is done")
#                 insights = job.get_result(params={"limit": 1000})
#                 break
#         if len(insights) == 0:
#             logger.info('No data from %s to %s', date_from, date_to)
#             telegram.info('⏹No data from %s to %s', date_from, date_to)
#
#             return 'ok'
#
#     except Exception as e:
#         logger.info('Got error: %s', e)
#         telegram.error('❗Got error: %s', e)
#         raise
#
#     fb_source = []
#
#     for _, item in enumerate(insights):
#
#         actions = []
#         conversions = []
#
#         if 'actions' in item:
#             for i, value in enumerate(item['actions']):
#                 actions.append(
#                     {'action_type': value['action_type'], 'value': value['value']})
#
#         if 'conversions' in item:
#             for i, value in enumerate(item['conversions']):
#                 conversions.append(
#                     {'action_type': value['action_type'], 'value': value['value']})
#
#         fb_source.append({'date': item['date_start'],
#                           'account_id': item['account_id'],
#                           'ad_id': item['ad_id'],
#                           'ad_name': item['ad_name'],
#                           'adset_id': item['adset_id'],
#                           'adset_name': item['adset_name'],
#                           'campaign_id': item['campaign_id'],
#                           'campaign_name': item['campaign_name'],
#                           'clicks': item.get('clicks', 0),
#                           'impressions': item.get('impressions', 0),
#                           'spend': item.get('spend', 0),
#                           'conversions': conversions,
#                           'actions': actions,
#                           'country': country
#                           })
#
#     row_count = len(fb_source)
#
#     json_data = {
#         'source': tg_name,
#         'customer_id': account_id,
#         'date': get_date(),
#         'table_id': table_id,
#         'report': "default",
#         'num_rows': row_count
#     }
#
#     if exist_dataset_table(bigquery_client, table_id, dataset_id, project_id, schema_facebook_stat,
#                            clustering_fields_facebook) == 'ok':
#
#         try:
#
#             insert_json_bq(bigquery_client, table_id, dataset_id,
#                            project_id, fb_source, schema_facebook_stat)
#             if exist_dataset_table(bigquery_client, row_table_id, row_dataset_id, project_id, schema_rows) == 'ok':
#
#                 try:
#                     insert_json_bq(bigquery_client, row_table_id, row_dataset_id, project_id, [
#                         json_data], schema_rows)
#                 except Exception as e:
#                     logger.error('Got error: %s', e)
#                     telegram.error('⛔️Got error: %s', e)
#                     exit(0)
#
#             logger.info(
#                 'Transfered data to Bigquery tables: from %s to %s', date_from, date_to)
#             telegram.info(
#                 '✅Transfered data to Bigquery tables: from %s to %s', date_from, date_to)
#
#         except Exception as e:
#             logger.error('Got error: %s', e)
#             telegram.error('❗Got error: %s', e)
#
#         return 'ok'



# **Final function**

In [ ]:
def get_facebook_data(event, context):
    '''
    Final function
    '''

    # pubsub_message = base64.b64decode(event['data']).decode('utf-8')
    pubsub_message = event['attributes']['data']

    credentials = service_account.Credentials.from_service_account_file("/content/intertop-ukraine-7924e-bda8050b5d2d.json")
    bigquery_client = bigquery.Client(credentials=credentials)

    tg_name = event['attributes']['tg_name']
    tg_users = event['attributes']['tg_users'].split(',')
    tg_users = [int(i) for i in tg_users]
    tg_token = event['attributes']['tg_token']

    telegram = logging.getLogger(tg_name)
    tg_logger.setup(telegram, token=tg_token, users=tg_users)

    row_dataset_id = event['attributes']['row_dataset_id']
    row_table_id = event['attributes']['row_table_id']

    if 'startdate' in event['attributes']:
        date_from = event['attributes']['startdate']
    else:
        date_from = get_date()

    if 'enddate' in event['attributes']:
        date_to = event['attributes']['enddate']
    else:
        date_to = get_date()

    if pubsub_message == 'get_currency':

        table_id = event['attributes']['table_id']
        dataset_id = event['attributes']['dataset_id']
        project_id = event['attributes']['project_id']

        api_key = event['attributes']['api_key']
        from_currency = event['attributes']['from_currency']
        to_currency = event['attributes']['to_currency']

        cur_source = []

        params = {'currencies': to_currency,
                  'source': from_currency,
                  'start_date': date_from,
                  'end_date': date_to
                  }

        headers = {'apikey': api_key}

        url = 'https://api.apilayer.com/currency_data/timeframe'

        try:
            r = requests.get(url, params=params, headers=headers, timeout=20)
        except requests.exceptions.RequestException as e:
            logger.error('Request to currencylayer error: %s', e)
            return e

        if r.json()["success"] is True:

            exist_dataset_table(bigquery_client, table_id,
                                dataset_id, project_id, schema_exchange_rate)

            if r.json()["success"] is True:

                exist_dataset_table(bigquery_client, table_id, dataset_id, project_id, schema_exchange_rate)

                for date_key, currencies in r.json()['quotes'].items():
                    currency_data = {'date': date_key}
                    currency_data.update(currencies)
                    cur_source.append(currency_data)

            row_count = len(cur_source)
            json_data = {
                'date': get_date(),
                'source': tg_name,
                'customer_id': project_id,
                'table_id': table_id,
                'report': "default",
                'num_rows': row_count
            }

            insert_json_bq(bigquery_client, table_id, dataset_id,
                           project_id, cur_source, schema_exchange_rate)

            if exist_dataset_table(bigquery_client, row_table_id, row_dataset_id, project_id, schema_rows) == 'ok':

                try:
                    insert_json_bq(bigquery_client, row_table_id, row_dataset_id, project_id, [
                        json_data], schema_rows)
                except Exception as e:
                    logger.error('Got error: %s', e)
                    telegram.error('⛔️Got error: %s', e)
                    exit(0)

            logger.info(
                'Currency data was transferred to Bigquery tables: from %s to %s', date_from, date_to)
            telegram.info(
                '✅Currency data was transferred to Bigquery tables: from %s to %s', date_from, date_to)

        else:
            logger.error('Request to currencylayer error: %s',
                         r.json()["error"]["info"])

        return 'ok'


    if pubsub_message == 'get_facebook':

        table_id = event['attributes']['table_id']
        dataset_id = event['attributes']['dataset_id']
        project_id = event['attributes']['project_id']

        app_id = event['attributes']['app_id']
        app_secret = event['attributes']['app_secret']
        access_token = event['attributes']['access_token']
        account_id = event['attributes']['account_id']
        country = event['attributes']['country']

        retry_attempts = 3
        retry_delay = 60
        for attempt in range(retry_attempts):
            try:
                FacebookAdsApi.init(app_id, app_secret, access_token, api_version='v20.0')
                account = AdAccount('act_' + str(account_id))

                job = account.get_insights(fields=[
                    AdsInsights.Field.account_id,
                    AdsInsights.Field.campaign_id,
                    AdsInsights.Field.campaign_name,
                    AdsInsights.Field.adset_name,
                    AdsInsights.Field.adset_id,
                    AdsInsights.Field.ad_name,
                    AdsInsights.Field.ad_id,
                    AdsInsights.Field.spend,
                    AdsInsights.Field.impressions,
                    AdsInsights.Field.clicks,
                    AdsInsights.Field.actions,
                    AdsInsights.Field.conversions
                ], params={
                    'level': 'ad',
                    'time_range': {
                        'since': date_from,
                        'until': date_to
                    },
                    'time_increment': 1
                }, is_async=True)

                while True:
                    job = job.api_get()
                    time.sleep(1)
                    if job[AdReportRun.Field.async_status] == 'Job Completed':
                        logger.info("Async job is done")
                        insights = job.get_result(params={"limit": 1000})
                        break

                if len(insights) == 0:
                    logger.info('No data from %s to %s', date_from, date_to)
                    telegram.info('⏹No data from %s to %s', date_from, date_to)
                    return 'ok'
                break

            except Exception as e:
                error_message = str(e)
                if "code: 1" in error_message and "error_subcode: 99" in error_message:
                    logger.warning(f"Trying {attempt + 1} from {retry_attempts}. Repeat after {retry_delay} seconds.")
                    telegram.warning(f"⚠️Trying {attempt + 1} from {retry_attempts}. Repeat after {retry_delay} seconds.")
                    time.sleep(retry_delay)
                else:
                    logger.error('Got error: %s', e)
                    telegram.error('❗Got error: %s', e)
                    raise

        else:
            logger.error('❗Three attempts failed.')
            telegram.error('❗Three attempts failed.')
            raise Exception("❗Three attempts failed.")

        fb_source = []

        for _, item in enumerate(insights):

            actions = []
            conversions = []

            if 'actions' in item:
                for i, value in enumerate(item['actions']):
                    actions.append(
                        {'action_type': value['action_type'], 'value': value['value']})

            if 'conversions' in item:
                for i, value in enumerate(item['conversions']):
                    conversions.append(
                        {'action_type': value['action_type'], 'value': value['value']})

            fb_source.append({'date': item['date_start'],
                              'account_id': item['account_id'],
                              'ad_id': item['ad_id'],
                              'ad_name': item['ad_name'],
                              'adset_id': item['adset_id'],
                              'adset_name': item['adset_name'],
                              'campaign_id': item['campaign_id'],
                              'campaign_name': item['campaign_name'],
                              'clicks': item.get('clicks', 0),
                              'impressions': item.get('impressions', 0),
                              'spend': item.get('spend', 0),
                              'conversions': conversions,
                              'actions': actions,
                              'country': country
                              })

        row_count = len(fb_source)

        json_data = {
            'source': tg_name,
            'customer_id': account_id,
            'date': get_date(),
            'table_id': table_id,
            'report': "default",
            'num_rows': row_count
        }

        if exist_dataset_table(bigquery_client, table_id, dataset_id, project_id, schema_facebook_stat,
                               clustering_fields_facebook) == 'ok':

            try:
                insert_json_bq(bigquery_client, table_id, dataset_id,
                               project_id, fb_source, schema_facebook_stat)
                if exist_dataset_table(bigquery_client, row_table_id, row_dataset_id, project_id, schema_rows) == 'ok':

                    try:
                        insert_json_bq(bigquery_client, row_table_id, row_dataset_id, project_id, [
                            json_data], schema_rows)
                    except Exception as e:
                        logger.error('Got error: %s', e)
                        telegram.error('⛔️Got error: %s', e)
                        exit(0)

                logger.info(
                    'Transfered data to Bigquery tables: from %s to %s', date_from, date_to)
                telegram.info(
                    '✅Transfered data to Bigquery tables: from %s to %s', date_from, date_to)

            except Exception as e:
                logger.error('Got error: %s', e)
                telegram.error('❗Got error: %s', e)

        return 'ok'


# **Get FB data and upload to BQ**

In [ ]:
event = {
    'attributes': {
        'data': 'get_facebook',
        'access_token': 'EAAUTizf8tZBEBO8xSeAoOm7FgHWCNgXTyQUhshxIogzCSKM7KYUrV28N9xm0oVZAofmWvgaeJNBKUFfuZCmI3L43x2aA6mSVKokx2j7xhBZBFk9uV55MeAAFtT7eZCvIfi8qBfO8TCZCFCCLJMZBpLq2iZAqdhOJJSMDwxA5ZBF1gRvJbRxfZAR25FyKhF6rjAQRZC9',
        'account_id': '690871802206027',
        'app_id': '1428863544244193',
        'app_secret': '4f75f58d8e7ad8b79b57c7c8398a7836',
        'country': 'UA',
        'dataset_id': 'intertop_db',
        'project_id': 'gtm-txdvwmd-ztyzm',
        'row_dataset_id': 'logs',
        'row_table_id': 'count_rows',
        'table_id': 'raw_facebook_1',
        'tg_name': 'Facebook:690871802206027',
        'tg_token': '5880247367:AAE7tiEy8xNvoYyxa98-pmfBIIDlSEDLlAg',
        'tg_users': '7429469443', # ,362954599,5528074955,725429685,388246593
        'startdate': '2025-03-01',
        'enddate': '2025-03-04'
    }
}
context = {}

get_facebook_data(event, context)

INFO:root:Async job is done
INFO:root:Date for reports was generated
INFO:root:Job config was prepared
INFO:root:Data was uploaded to the table raw_facebook_1
INFO:root:Job config was prepared
INFO:root:Data was uploaded to the table count_rows
INFO:root:Transfered data to Bigquery tables: from 2025-01-10 to 2025-01-13
ERROR:tg_logger.handler:Exception while sending <b>Facebook:690871802206027:INFO</b> - <code>✅Transfered data to Bigquery tables: from 2025-01-10 to 2025-01-13</code> to 362954599:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/tg_logger/handler.py", line 39, in emit
    self.bot.send_message(user_id, msg, parse_mode="HTML")
  File "/usr/local/lib/python3.11/dist-packages/telebot/__init__.py", line 1800, in send_message
    apihelper.send_message(
  File "/usr/local/lib/python3.11/dist-packages/telebot/apihelper.py", line 275, in send_message
    return _make_request(token, method_url, params=payload, method='post')
           ^^^^^^^^

'ok'